# Notebook 1: Prerequisites & Setup
## Deploy Your First Azure AI Agent Service App on Azure App Service

**Source:** [Microsoft Tech Community Blog](https://techcommunity.microsoft.com/blog/azure-ai-foundry-blog/deploy-your-first-azure-ai-agent-service-powered-app-on-azure-app-service/4396173) by Robert Rita, AI Cloud Solution Architect, ASEAN

---

This notebook covers:
1. Prerequisites checklist
2. Azure AI Hub & Project creation
3. Model deployment (GPT-4o)
4. Agent creation & configuration
5. Local environment setup
6. Environment variables configuration

## Architecture Overview

The solution uses the following Azure services working together:

```
┌─────────────────────────────────────────────────────────┐
│                    Azure Cloud                          │
│                                                         │
│  ┌──────────────────┐     ┌──────────────────────────┐  │
│  │  Azure App        │     │  Azure AI Foundry Hub     │  │
│  │  Service           │────▶│                          │  │
│  │  (Chainlit App)   │     │  ┌────────────────────┐  │  │
│  │                   │     │  │  AI Foundry Project │  │  │
│  └──────────────────┘     │  │                    │  │  │
│         │                  │  │  ┌──────────────┐ │  │  │
│         │ Managed          │  │  │  Agent        │ │  │  │
│         │ Identity         │  │  │  (GPT-4o)    │ │  │  │
│         │                  │  │  └──────────────┘ │  │  │
│         ▼                  │  └────────────────────┘  │  │
│  ┌──────────────────┐     │                          │  │
│  │  Azure OpenAI     │◀────│  Models + Endpoints      │  │
│  │  (gpt-4o model)  │     └──────────────────────────┘  │
│  └──────────────────┘                                   │
└─────────────────────────────────────────────────────────┘
```

**Key components:**
- **Azure AI Foundry Hub** — Central management for agents and models
- **Azure OpenAI (GPT-4o)** — Language model powering agent intelligence
- **Chainlit Application** — Python-based conversational UI layer
- **Azure App Service** — Hosting and auto-scaling infrastructure
- **Managed Identity** — Secure authentication without credential storage

---
## Step 1: Prerequisites Checklist

Before starting, make sure you have the following:

| Requirement | Description | Status |
|---|---|---|
| **Azure Subscription** | Active Azure subscription ([Create free account](https://azure.microsoft.com/free/)) | [ ] |
| **Azure AI Foundry Access** | Permissions to create hubs and projects | [ ] |
| **Azure CLI** | Command-line tool for Azure resource management | [ ] |
| **Python 3.12+** | Python runtime (3.12 or higher) | [ ] |
| **Git** | For cloning the sample repository | [ ] |
| **Code Editor** | Visual Studio Code recommended | [ ] |
| **Azure App Service basics** | Familiarity with resource groups and hosting plans | [ ] |

### 1.1 Verify local prerequisites

Run the cells below to check your local environment is ready.

In [ ]:
# Check Python version (must be 3.12+)
import sys
print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 12), "Python 3.12 or higher is required!"
print("Python version check passed.")

In [ ]:
# Check Azure CLI is installed
import subprocess

result = subprocess.run(["az", "--version"], capture_output=True, text=True)
if result.returncode == 0:
    first_line = result.stdout.strip().split("\n")[0]
    print(f"Azure CLI installed: {first_line}")
else:
    print("Azure CLI is NOT installed.")
    print("Install it from: https://learn.microsoft.com/en-us/cli/azure/install-azure-cli")

In [ ]:
# Check Git is installed
result = subprocess.run(["git", "--version"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"Git installed: {result.stdout.strip()}")
else:
    print("Git is NOT installed.")
    print("Install it from: https://git-scm.com/downloads")

---
## Step 2: Azure AI Hub & Project Creation

### 2.1 Create an Azure AI Foundry Hub

1. Sign in to the [Azure Portal](https://portal.azure.com)
2. Search for **"AI Foundry"** in the top search bar
3. Click **Create** and select **Hub**
4. Fill in the details:
   - **Subscription**: Select your Azure subscription
   - **Resource Group**: Create new or use existing
   - **Region**: Choose your preferred region
   - **Name**: Give your hub a unique name
   - **Connect AI Services**: Link to Azure AI Services
5. Click **Review + Create**, then **Create**
6. Once deployed, click **"Launch Azure AI Foundry"**

### 2.2 Create a Project under the Hub

1. Inside Azure AI Foundry, click **"+ New project"**
2. Give it a name (e.g., `my-agent-project`)
3. Select your hub
4. Click **Create**

> **Important:** Note down your **Project Connection String** from the project's **Overview** page. You'll need it later.

---
## Step 3: Deploy the GPT-4o Model

1. In your AI Foundry project, navigate to **"Models + Endpoints"** in the left panel
2. Click **"Deploy model"** then **"Deploy base model"**
3. Select **gpt-4o** from the model list
4. Click **Confirm**
5. Accept the default settings and click **Deploy**

> The model will be deployed and connected to your project automatically.

---
## Step 4: Create an AI Agent

1. In your AI Foundry project, go to the **"Agents"** section
2. Click **"+ New Agent"**
3. Configure the agent:
   - **Name**: Give it a descriptive name
   - **Model**: Select the deployed **gpt-4o**
   - **Instructions**: Set the system prompt, for example:
     ```
     You are a helpful assistant capable of answering queries and performing tasks.
     ```
4. **(Optional)** Add tools:
   - **Code Interpreter** — Execute code for data analysis
   - **OpenAPI Tools** — Interact with external APIs
5. Click **Create**

> **Important:** Note down your **Agent ID** from the Agents section. You'll need this along with the connection string.

---
## Step 5: Clone the Sample Repository & Set Up Local Environment

In [ ]:
# Clone the sample repository
!git clone -b Deploy-AI-Agent-App-Service https://github.com/robrita/tech-blogs tech-blogs-agent

In [ ]:
# Install required Python packages
!pip install azure-ai-projects azure-identity chainlit python-dotenv

In [ ]:
# Verify the key packages are importable
try:
    import azure.ai.projects
    print(f"azure-ai-projects installed successfully")
except ImportError:
    print("ERROR: azure-ai-projects not found")

try:
    import azure.identity
    print(f"azure-identity installed successfully")
except ImportError:
    print("ERROR: azure-identity not found")

try:
    import chainlit
    print(f"chainlit installed successfully")
except ImportError:
    print("ERROR: chainlit not found")

try:
    import dotenv
    print(f"python-dotenv installed successfully")
except ImportError:
    print("ERROR: python-dotenv not found")

---
## Step 6: Configure Environment Variables

You need two key values from your Azure AI Foundry project:

| Variable | Where to Find It |
|---|---|
| `AIPROJECT_CONNECTION_STRING` | AI Foundry Project > **Overview** page |
| `AGENT_ID` | AI Foundry Project > **Agents** section |

### 6.1 Create the `.env` file

Run the cell below, then **edit the generated `.env` file** with your actual values.

In [ ]:
# Create a .env file with placeholder values
# IMPORTANT: Replace the placeholder values with your actual credentials

env_content = """# Azure AI Agent Service Configuration
# Get AIPROJECT_CONNECTION_STRING from: AI Foundry Project > Overview
AIPROJECT_CONNECTION_STRING=your_connection_string_here

# Get AGENT_ID from: AI Foundry Project > Agents section
AGENT_ID=your_agent_id_here
"""

env_path = ".env"

import os
if os.path.exists(env_path):
    print(f"WARNING: {env_path} already exists. Not overwriting.")
    print("Delete it manually if you want to regenerate.")
else:
    with open(env_path, "w") as f:
        f.write(env_content)
    print(f"Created {env_path}")
    print("NEXT STEP: Open .env and replace placeholder values with your actual credentials.")

### 6.2 Validate environment variables are loaded

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

connection_string = os.getenv("AIPROJECT_CONNECTION_STRING")
agent_id = os.getenv("AGENT_ID")

# Validate (without printing actual secrets)
if connection_string and connection_string != "your_connection_string_here":
    print(f"AIPROJECT_CONNECTION_STRING: SET (starts with '{connection_string[:20]}...')")
else:
    print("WARNING: AIPROJECT_CONNECTION_STRING is not set or still has placeholder value!")

if agent_id and agent_id != "your_agent_id_here":
    print(f"AGENT_ID: SET (starts with '{agent_id[:10]}...')")
else:
    print("WARNING: AGENT_ID is not set or still has placeholder value!")

---
## Step 7: Authenticate with Azure

Before running the app, log in to Azure so `DefaultAzureCredential` can authenticate.

In [ ]:
# Log in to Azure (this will open a browser window)
!az login

In [ ]:
# Verify Azure login and test the connection to AI Foundry
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

try:
    project_client = AIProjectClient.from_connection_string(
        conn_str=os.getenv("AIPROJECT_CONNECTION_STRING"),
        credential=DefaultAzureCredential()
    )
    print("Successfully connected to Azure AI Foundry project!")
    print("Project client is ready.")
except Exception as e:
    print(f"Connection failed: {e}")
    print("\nTroubleshooting:")
    print("1. Make sure you ran 'az login' successfully")
    print("2. Verify your AIPROJECT_CONNECTION_STRING in .env")
    print("3. Ensure you have access to the AI Foundry project")

---
## Setup Complete!

At this point you should have:

- [x] All local prerequisites verified (Python 3.12+, Azure CLI, Git)
- [x] Azure AI Foundry Hub created
- [x] AI Foundry Project created under the hub
- [x] GPT-4o model deployed
- [x] AI Agent created and configured
- [x] Sample repository cloned
- [x] Python packages installed
- [x] `.env` file configured with your credentials
- [x] Azure login successful and connection verified

### What's Next?

In the **next notebook**, we'll cover:
- The Chainlit application code walkthrough
- Deploying to Azure App Service
- Configuring Managed Identity and IAM roles
- Testing the deployed application

### Useful Links

- [Azure AI Agent Service Documentation](https://learn.microsoft.com/en-us/azure/ai-services/agents/overview)
- [Sample Repository](https://github.com/robrita/tech-blogs/tree/Deploy-AI-Agent-App-Service)
- [Azure App Service Documentation](https://learn.microsoft.com/en-us/azure/app-service/)
- [Chainlit Documentation](https://docs.chainlit.io/)